# <span style="color:darkblue"> Assignment 4: Textual Analysis of Bills and Actions from 116th Congress of the United States  </span>

<font size = 4>
By: Tanya Jagdish

<b> Import libraries

In [103]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from wordcloud import WordCloud, STOPWORDS

<b> Load dataset

In [104]:
bills_actions = pd.read_csv("data_raw/bills_actions.csv")

##  <span style="color:darkblue"> <b> Understanding the data + basic descriptive statistics

<b> <i> How many distinct "bill" identifiers are there? </b>

In [105]:
bills_actions.dtypes
n_bills = bills_actions["bill_number"].nunique()
n_rows = len(bills_actions)
print(f"Distinct # of bills: {n_bills:,} | Total rows in dataset: {n_rows:,}")

Distinct # of bills: 1,331 | Total rows in dataset: 3,303


<b> <i> What proportion of the rows belong to different categories? </b>

In [106]:
category_proportions = (bills_actions["category"]
                        .value_counts(normalize = True)
                        .mul(100).round(2)
                        .rename("% of rows"))

category_proportions

category
amendment                       46.29
house bill                      27.31
senate bill                     15.56
house resolution                 7.08
senate resolution                1.82
house joint resolution           0.67
house concurrent resolution      0.61
senate concurrent resolution     0.42
senate joint resolution          0.24
Name: % of rows, dtype: float64

<b> <i> Within each category, what's the proportion of rows that belongs to different “main_action”? </b>

In [107]:
#count number of main_action rows within each category
main_action_by_cat = (bills_actions
                      .groupby("category")["main_action"]
                      .value_counts()
                      .rename("Count of rows")
                      .reset_index()
)

#calculate proportion
main_action_by_cat["% of main action within each category"] = (main_action_by_cat
                             .groupby("category")["Count of rows"]
                             .transform(lambda x: x / x.sum()*100)
                             .round(2))

#pretty formatting
main_action_by_cat.style.hide(axis="index").format({"Count of rows": "{:,.0f}", "% of main action within each category": "{:.2f}%"})

category,main_action,Count of rows,% of main action within each category
amendment,house amendment offered,828,54.15%
amendment,other house amendment actions,397,25.96%
amendment,senate amendment proposed (on the floor),286,18.71%
amendment,other senate amendment actions,18,1.18%
house bill,house floor actions,702,77.83%
house bill,house committee/subcommittee actions,132,14.63%
house bill,senate committee/subcommittee actions,54,5.99%
house bill,resolving differences -- house actions,6,0.67%
house bill,senate floor actions,5,0.55%
house bill,resolving differences -- senate actions,3,0.33%


##  <span style="color:darkblue"> <b>House floor actions on Senate bills

To understand cross-chamber dynamic, we are looking at House floor actions on Senate bills. 

A few conceptual notes on the data:

* A senate bill originates in the Senate

* But once a bill passes the Senate, it moves to the House, where it can be debated, ammended, or voted on

* All these steps (except ammendment) are recorded as 'House floor actions' in the column 'main_action'

Given this, I first calculate the number of house floor actions on senate bills

### Number of house floor actions on senate bills

In [108]:
# filter for Senate bills and House floor actions
house_floor_in_senate = bills_actions.query(
    'category == "senate bill" and main_action == "house floor actions"'
)

# number of such rows
total_actions = len(house_floor_in_senate)
print(f"Number of House floor actions on Senate bills: {total_actions:,}")


Number of House floor actions on Senate bills: 116


### Proportion of these actions that were suspended and suspended as ammended

In [109]:
# prop of bills suspended
suspended = house_floor_in_senate["action"].str.contains("suspend", case = False, na = False)
prop_suspended = suspended.mean()*100

# prop of bills suspended as ammended
suspended_amended = house_floor_in_senate["action"].str.contains("suspend", case=False, na=False) & \
                    house_floor_in_senate["action"].str.contains("amend", case=False, na=False)
prop_suspended_amended = suspended_amended.mean() * 100

#display results
print(f"Total House floor actions on Senate bills: {total_actions:,}")
print(f"Proportion suspended: {prop_suspended:.2f}%")
print(f"Proportion suspended as amended: {prop_suspended_amended:.2f}%")

Total House floor actions on Senate bills: 116
Proportion suspended: 68.97%
Proportion suspended as amended: 15.52%


### Names of legislators mentioned in House floor actions

Conceptually, in this section, we are looking at which legislators are mentioned in the House floor action text (in column 'action' within the dataset)

For instance, if the action column says "On motion of Mr. Smith, the rules were suspended and the bill passed," we want to try to pull out "Smith" from the text

To do this, we will:

1. Use regular expression patterns to pull out all names that follow "Mr.", "Mrs.", "Ms."

2. Expand those into a dataset of person-bill pairs

3. Count how many distinct bills each person is mentioned in (e.g., if "Mr. Smith" appears in 3 different bills, count = 3)

4. Visualize or describe the distribution (e.g., many people mentioned once, few mentioned multiple times, etc.)

In [110]:
#string pattern
pattern = r"(?i)\b(?:Mrs|Ms|Mr)\.?\s*(\S+)"

#sanity check - making sure code works
tests = [
    "Mrs. Smith moved to suspend the rules.",
    "Mr Jones spoke.",
    "ms O'Neill objected.",
    "MRS  De-Lauro offered an amendment."
]

#pd.Series(tests).str.findall(pattern)


Note on the string pattern chosen above

* \b - for word boundary

* (?i) - case insensitive (e.g., matches Mr. or mr.)

* (?:mrs|ms|mr) - a non-capturing group matching one of the titles. I intentionally choose to order it as 'mrs|ms|mr' to search for 'mrs' first and then 'mr'

* \ . ? - an optional dot after the title (e.g., matches Mr. or mr)

* \s* - zero or more spaces between title and the name

* (\S+) - capturing group of one or more non-space characters

In [ ]:
#search for names
names = house_floor_in_senate["action"].str.findall(pattern)

#add names column to dataset
house_floor_in_senate["person"] = names

#convert list of names to string of names
house_floor_in_senate["person"] = house_floor_in_senate["person"].str[0]

#remove rows with no person and keep only person, bill_number column
bill_person_pair = house_floor_in_senate.dropna(subset = ["person"])[["bill_number", "person"]]

#house_floor_in_senate.head()
#bill_person_pair.head()

C:\Users\tanya\AppData\Local\Temp\ipykernel_33864\336965835.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  house_floor_in_senate["person"] = names
C:\Users\tanya\AppData\Local\Temp\ipykernel_33864\336965835.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  house_floor_in_senate["person"] = house_floor_in_senate["person"].str[0]


,bill_number,person
559,1436,Lofgren
560,1693,Evans
562,1749,Clay
563,2047,DeGette
564,2249,Larsen
